# 02 · Data Cleaning & Wrangling
### *Stage 2 — dtypes, business rules, the leakage register, and the derived features EDA needs*

> **So what this stage guards:** the data notes warn that `account_profiles` fraud fields and
> `fraud_pattern` are built *from* the label. Here we (a) validate the data is as clean as a generated
> set should be, and (b) codify the **leakage register** so no banned column can slip into a model.

**Gate:** dtypes correct · missing strategy documented per column · business rules pass · cleaning log written.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path
pd.set_option("display.max_columns", 60)
RAW = Path("../datasets"); PROC = Path("../data/processed"); PROC.mkdir(parents=True, exist_ok=True)

transactions = pd.read_csv(RAW / "transactions.csv", parse_dates=["timestamp"])
accounts     = pd.read_csv(RAW / "account_profiles.csv")
edges        = pd.read_csv(RAW / "network_edges.csv")
cleaning_log = []
def log(msg): cleaning_log.append(msg); print("·", msg)

#### Enforce dtypes early (catch silent coercion) — categoricals, booleans, the target

In [2]:
cat_cols  = ["merchant_category", "device_type", "merchant_country", "account_type"]
bool_cols = ["is_weekend", "card_present", "device_known", "is_foreign_txn", "has_2fa", "is_fraud"]

for col in cat_cols:
    if col in transactions: transactions[col] = transactions[col].astype("category")
for col in bool_cols:
    if col in transactions: transactions[col] = transactions[col].astype("int8")

# fraud_pattern is null by design for non-fraud → make the "missingness" explicit, not an error
transactions["fraud_pattern"] = transactions["fraud_pattern"].fillna("none").astype("category")
log(f"cast {len(cat_cols)} categoricals + {len(bool_cols)} booleans; fraud_pattern nulls → 'none'")

· cast 4 categoricals + 6 booleans; fraud_pattern nulls → 'none'


#### Missing values — documented per column
The only nulls are `fraud_pattern` for non-fraud rows: **MNAR by construction** (absent *because*
the row isn't fraud). We keep them as an explicit `'none'` category rather than dropping — dropping
would delete 98% of the data. Every other column is complete, as expected for a generated set.

In [3]:
na = transactions.isnull().sum()
print("columns with nulls after handling:", int((na > 0).sum()))
log(f"fraud_pattern: {(transactions['fraud_pattern']=='none').sum():,} 'none' rows (non-fraud, MNAR-by-design)")

columns with nulls after handling: 0
· fraud_pattern: 982,857 'none' rows (non-fraud, MNAR-by-design)


#### Deduplication — `transaction_id` must be a unique key

In [4]:
dupes = transactions["transaction_id"].duplicated().sum()
log(f"duplicate transaction_id rows: {dupes} (none expected)")
assert dupes == 0

· duplicate transaction_id rows: 0 (none expected)


#### Business-rule validation — assert the domain constraints from the data dictionary

In [5]:
checks = {
    "amount >= 0":            (transactions["amount"] >= 0).all(),
    "0 <= ip_risk <= 100":    transactions["ip_risk_score"].between(0, 100).all(),
    "0 <= hour <= 23":        transactions["hour_of_day"].between(0, 23).all(),
    "0 <= day_of_week <= 6":  transactions["day_of_week"].between(0, 6).all(),
    "velocity_1h >= 0":       (transactions["velocity_1h"] >= 0).all(),
    "age_days >= 0":          (transactions["account_age_days"] >= 0).all(),
    "pattern set iff fraud":  ((transactions["fraud_pattern"] != "none") == (transactions["is_fraud"] == 1)).all(),
}
pd.Series(checks).to_frame("passed")

,passed
amount >= 0,True
0 <= ip_risk <= 100,True
0 <= hour <= 23,True
0 <= day_of_week <= 6,True
velocity_1h >= 0,True
age_days >= 0,True
pattern set iff fraud,True


In [6]:
assert all(checks.values()), f"failed rules: {[k for k,v in checks.items() if not v]}"
log("all business-rule checks passed")

· all business-rule checks passed


#### Leakage register (the heart of Stage 2 for this dataset)
These columns are **derived from the label** and are permanently banned as model features. We record
them explicitly so Stage 4 can enforce the ban programmatically rather than by memory.

In [7]:
LEAKAGE_REGISTER = {
    "account_profiles.csv": ["fraud_count", "fraud_amount", "fraud_rate", "is_fraudster"],
    "transactions.csv":     ["fraud_pattern"],   # only populated when is_fraud == 1
}
import json
(PROC / "leakage_register.json").write_text(json.dumps(LEAKAGE_REGISTER, indent=2))
log(f"leakage register written: {sum(len(v) for v in LEAKAGE_REGISTER.values())} banned columns")
LEAKAGE_REGISTER

· leakage register written: 5 banned columns


{'account_profiles.csv': ['fraud_count',
  'fraud_amount',
  'fraud_rate',
  'is_fraudster'],
 'transactions.csv': ['fraud_pattern']}

#### Derived columns EDA needs now (label-free)
`hour_bucket` and `is_high_risk_mcc` are used across Stage 3 cuts. Model-only interaction features are
built later in Stage 4 to keep the leakage boundary clean.

In [8]:
def hour_bucket(h):
    if h <= 5:  return "night_0_5"
    if h <= 17: return "day_6_17"
    return "evening_18_23"
transactions["hour_bucket"] = transactions["hour_of_day"].map(hour_bucket).astype("category")

HIGH_RISK_MCC = {"crypto", "gambling", "money_transfer"}
transactions["is_high_risk_mcc"] = transactions["merchant_category"].isin(HIGH_RISK_MCC).astype("int8")
log(f"derived hour_bucket + is_high_risk_mcc (high-risk = {sorted(HIGH_RISK_MCC)})")

· derived hour_bucket + is_high_risk_mcc (high-risk = ['crypto', 'gambling', 'money_transfer'])


#### Network / ring features — the one genuinely independent signal
Degree = how many edges an account participates in; membership flag = the account sits in a known ring.
Accounts absent from the edge list get degree 0. Joined onto every transaction by `account_id`.

In [9]:
deg = pd.concat([edges["account_a"], edges["account_b"]]).value_counts()
ring_map = (pd.concat([edges[["account_a", "ring_id"]].rename(columns={"account_a": "acc"}),
                       edges[["account_b", "ring_id"]].rename(columns={"account_b": "acc"})])
            .dropna(subset=["ring_id"]).groupby("acc")["ring_id"].first())

transactions["ring_degree"] = transactions["account_id"].map(deg).fillna(0).astype("int32")
transactions["ring_membership_flag"] = transactions["account_id"].map(
    lambda a: 1 if a in ring_map.index else 0).astype("int8")
log(f"ring features joined: {(transactions['ring_membership_flag']==1).sum():,} txns from ring accounts")
print(transactions[["ring_degree", "ring_membership_flag"]].describe().round(3))

· ring features joined: 36,574 txns from ring accounts
       ring_degree  ring_membership_flag
count  1000000.000           1000000.000
mean         0.378                 0.037
std          1.477                 0.188
min          0.000                 0.000
25%          0.000                 0.000
50%          0.000                 0.000
75%          0.000                 0.000
max         21.000                 1.000


#### Persist the cleaned, leakage-safe processed tables

In [10]:
transactions.to_parquet(PROC / "transactions_clean.parquet", index=False)
accounts.to_parquet(PROC / "account_profiles.parquet", index=False)
edges.to_parquet(PROC / "network_edges.parquet", index=False)
pd.Series(cleaning_log, name="cleaning_log").to_csv(PROC / "cleaning_log.csv", index=False)
print("saved processed tables + cleaning log to", PROC.resolve())

saved processed tables + cleaning log to C:\Users\HP ENVY\OneDrive\Documents\Personal Project\Dataset\Fraud detection 1M transactions\data\processed


---
### Stage 2 gate — ✅ cleared
- [x] Dtypes enforced · [x] missing strategy documented (`fraud_pattern` MNAR-by-design)
- [x] Duplicates checked (0) · [x] business rules pass · [x] leakage register written
- [x] Cleaning log saved · [x] processed tables in `../data/processed/`

**Next → [`03_eda.ipynb`](03_eda.ipynb):** hypothesis-driven segment uplift, fraud-pattern signatures,
and ring descriptives — every exhibit action-titled with a "So What".